# 06 — CNN rung 2: full-data transfer check, batch/LR pick, timing

**Decision this feeds** (`docs/superpowers/specs/2026-09-09-cnn-train-rung23-design.md`):
measures wall-clock time for one full-scale nested-CV fold (needed to
budget rung 3's 5-fold × 5-repeat run against the 2026-09-16 deadline),
picks a batch size/learning rate empirically among three candidates, and
runs one leave-one-family-out transfer check (holding out the 3.895mm
family, not the largest 2.46mm family — see the spec for why). **This
notebook does not decide the gate** — only `notebooks/07_cnn_rung3.ipynb`'s
full nested CV, scored against `model.build_combat_baseline()` = 0.5290
log loss, does that.

**Nested cross-validation**: every training run below splits its training
portion again, 90/10, to pick the early-stopping epoch — the row(s) being
scored (the outer test fold, or the held-out family) are never used for
training or stopping. This avoids the optimism bias of scoring on the
same data used for model selection.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers (timings, log loss values), not any per-row
output.

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels. Builds the shared
# on-disk volume cache reused by every training run below and in
# notebooks/07_cnn_rung3.ipynb.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
print("family counts:\n", labeled_df["inplane_family"].value_counts())

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
cache_build_seconds = time.time() - cache_start
print(f"cache build: {cache_build_seconds:.1f}s for {len(uids)} volumes "
      f"({cache_build_seconds / len(uids) * 1000:.1f} ms/volume) -- "
      f"this cost is paid once per notebook session, not once per fold.")

In [ ]:
# [RUN ME] (no data access itself -- defines a function used by the
# [RUN ME] cells below). num_workers=0 always: a cached dataset must not
# be handed to multiple DataLoader worker processes, each of which would
# rebuild its own copy of the cache and silently multiply wall-clock time.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    """Splits (train_uids, train_labels, train_family) 90/10 (stratified,
    `inner_splits` folds, fold 0) for early stopping, trains DatCNN,
    reloads the best checkpoint, and predicts on outer_uids -- which are
    never used for training or stopping. Returns
    (outer_probs, history, best_state); outer_probs is aligned to
    outer_uids' order (the outer loader is never shuffled)."""
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    outer_probs = np.concatenate(outer_probs)

    return outer_probs, history, best_state

In [ ]:
# [RUN ME] -- batch size / LR mini-experiment on outer fold 0 (Malladi et
# al. 2022 sqrt-LR-scaling candidate vs. an unscaled control vs. the
# rung-0/1 continuity control). Cheap at a warm cache -- RESOURCES.md.
outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

candidates = [(8, 1e-3, "rung0/1 continuity"), (32, 2e-3, "sqrt-LR-scaled"), (32, 1e-3, "unscaled control")]
batch_lr_results = {}
for batch_size, lr, tag in candidates:
    start = time.time()
    probs, history, best_state = train_and_score_nested(
        fold0_train_uids, fold0_train_labels, fold0_train_family,
        fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
    )
    elapsed = time.time() - start
    score = evaluate.log_loss_score(fold0_test_labels, probs)
    batch_lr_results[(batch_size, lr)] = {"tag": tag, "history": history, "score": score, "seconds": elapsed}
    torch.save(best_state, config.CHECKPOINT_DIR / f"rung2_fold0_batch{batch_size}_lr{lr:.0e}.pt")
    print(f"batch={batch_size:>3} lr={lr:.0e} ({tag:<20}): "
          f"inner-val best={min(history['val_loss']):.4f}, outer log loss={score:.4f}, "
          f"{elapsed:.1f}s ({elapsed / len(history['val_loss']):.2f}s/epoch)")

winner = min(batch_lr_results, key=lambda k: min(batch_lr_results[k]["history"]["val_loss"]))
print(f"\nwinner (lowest inner-validation log loss): batch={winner[0]}, lr={winner[1]:.0e}")
print("This run's winner IS the first of rung 3's 5 seed-repeats (same seed, same fold 0) -- "
      "notebooks/07_cnn_rung3.ipynb reuses this pairing rather than duplicating it.")

In [ ]:
# [RUN ME] -- leave-one-family-out transfer check. Holds out the 3.895mm
# family (not the largest family, 2.46mm) -- Wenzel et al. 2019 found the
# OTHER transfer direction (fine->coarse) already works well, so holding
# out 2.46mm would prove nothing (RESOURCES.md).
lofo_family = "3.895"
assert lofo_family in labeled_df["inplane_family"].unique(), \
    f"{lofo_family} not found -- check baseline_features.csv's inplane_family values"

held_out_mask = labeled_df["inplane_family"] == lofo_family
retained_mask = ~held_out_mask

retained_uids = labeled_df.loc[retained_mask, config.UID_COLUMN].tolist()
retained_labels = labeled_df.loc[retained_mask, config.TARGET_COLUMN].tolist()
retained_family = labeled_df.loc[retained_mask, "inplane_family"].tolist()
held_out_uids = labeled_df.loc[held_out_mask, config.UID_COLUMN].tolist()
held_out_labels = np.array(labeled_df.loc[held_out_mask, config.TARGET_COLUMN].tolist())

batch_size, lr = winner
lofo_probs, lofo_history, lofo_state = train_and_score_nested(
    retained_uids, retained_labels, retained_family,
    held_out_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
)
torch.save(lofo_state, config.CHECKPOINT_DIR / f"rung2_lofo_{lofo_family.replace('.', '_')}.pt")

# Three reference numbers on the SAME held-out rows -- a CNN number alone
# is uninterpretable, since the family's own base rate differs from the
# global 0.548458 (README.md's pairwise family tests).
family_base_rate = held_out_labels.mean()
base_rate_preds = np.full_like(held_out_labels, family_base_rate, dtype=float)
family_base_rate_logloss = evaluate.log_loss_score(held_out_labels, base_rate_preds)

baseline_feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
baseline_X_all = baseline_feat_df[["abs_asym", "striatal_ratio"]].to_numpy()
baseline_y_all = baseline_feat_df[config.TARGET_COLUMN].to_numpy()
baseline_family_all = baseline_feat_df["inplane_family"].to_numpy()

retained_baseline_mask = baseline_family_all != lofo_family
baseline_pipeline = model.build_combat_baseline()
baseline_pipeline.fit(baseline_X_all[retained_baseline_mask], baseline_y_all[retained_baseline_mask],
                       baseline_family_all[retained_baseline_mask])
held_out_baseline_mask = baseline_family_all == lofo_family
baseline_lofo_probs = baseline_pipeline.predict_proba(
    baseline_X_all[held_out_baseline_mask], baseline_family_all[held_out_baseline_mask])[:, 1]
baseline_lofo_logloss = evaluate.log_loss_score(baseline_y_all[held_out_baseline_mask], baseline_lofo_probs)

cnn_lofo_logloss = evaluate.log_loss_score(held_out_labels, lofo_probs)

print(f"LOFO family={lofo_family} (n={int(held_out_mask.sum())}):")
print(f"  family's own base-rate log loss: {family_base_rate_logloss:.4f} (base rate={family_base_rate:.3f})")
print(f"  build_combat_baseline() on these rows: {baseline_lofo_logloss:.4f}")
print(f"  CNN (nested, trained on retained families): {cnn_lofo_logloss:.4f}")

np.save(config.DATA_PROCESSED / f"rung2_lofo_{lofo_family.replace('.', '_')}_cnn_probs.npy", lofo_probs)

**What we're looking for:** how long does one full-scale nested-CV fold
take (to budget rung 3), which (batch, LR) candidate wins, and does the
CNN transfer to a held-out acquisition family (3.895mm) better than the
family's own base rate and better than the classical baseline restricted
to those same rows?

**What we found:** *(paste: cache-build seconds and ms/volume; the three
batch/LR candidates' inner-val best loss, outer fold log loss, and
seconds/epoch; the winner; the three LOFO reference numbers)*

**Decision / next step:** *(confirm the winning (batch, LR) to use in
notebooks/07_cnn_rung3.ipynb; note whether the LOFO CNN number beats both
its family's own base rate AND the classical baseline restricted to that
family -- if it doesn't beat the base rate, something is wrong with the
CNN pipeline at scale, worth investigating before running rung 3's full
5x5 CV)*